# Snow Segmentation Dataset Builder

This notebook automates the creation of a paired Sentinel-2 + Dynamic World dataset for semantic snow segmentation. It:

- samples ~4k tiles across curated high-altitude regions stored in `bounding_boxes.json`,
- downloads cloud-masked Sentinel-2 Level-1C imagery (13 spectral bands),
- fetches matching Dynamic World V1 labels for the same area/date windows,
- derives binary snow/no-snow masks from the semantic labels, and
- exports everything into `dataset/raw`, `dataset/all_labels`, and `dataset/snow_labels`.

Run the cells top-to-bottom once your Google Earth Engine (GEE) credentials are available. The code is structured so it can be re-run when additional tiles are needed or parameters change.

## 1. Environment setup

Install the dependencies from `requirements.txt` (Earth Engine API, geemap, rasterio, numpy, torch, etc.) inside your chosen Conda/virtual environment. This notebook only orchestrates exports and does not perform heavy ML training.

In [1]:

from __future__ import annotations

import csv
import json
import math
import random
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Dict, List, Tuple

import ee
import geemap
from tqdm.auto import tqdm

C:\Users\Domen\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:

# -----------------------------------------------------------------------------
# Global configuration
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "dataset"
RAW_DIR = DATASET_ROOT / "raw"
ALL_LABEL_DIR = DATASET_ROOT / "all_labels"
ALL_LABEL_PROB_DIR = DATASET_ROOT / "all_labels_probabilities"
SNOW_LABEL_DIR = DATASET_ROOT / "snow_labels"
MANIFEST_PATH = DATASET_ROOT / "manifest.csv"
FAILED_LOG = DATASET_ROOT / "failed_exports.json"
REGION_JSON = PROJECT_ROOT / "bounding_boxes.json"

# Sampling + export hyper-parameters
TARGET_TILE_COUNT = 20000
MIN_TILE_SIZE_KM = 0.5
MAX_TILE_SIZE_KM = 4.0
SENTINEL_INITIAL_WINDOW_DAYS = 10
SENTINEL_MAX_WINDOW_DAYS = 45
SENTINEL_WINDOW_STEP = 10
DYNAMIC_WORLD_INITIAL_WINDOW_DAYS = 10
DYNAMIC_WORLD_MAX_WINDOW_DAYS = 30
DYNAMIC_WORLD_WINDOW_STEP = 10
SENTINEL_CLOUD_THRESHOLD = 5  # percent
SNOW_CLASS_ID = 8
SNOW_MIN_PROB = 0.51
RANDOM_SEED = 1337
MAX_TILE_PIXELS = 1024
BASE_EXPORT_SCALE = 10

DATASET_DIRS = [RAW_DIR, ALL_LABEL_DIR, ALL_LABEL_PROB_DIR, SNOW_LABEL_DIR]

S2_BANDS = [
    "B1", "B2", "B3", "B4", "B5", "B6", "B7",
    "B8", "B8A", "B9", "B10", "B11", "B12",
]
DW_PROBABILITY_BANDS = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
]

for path in [DATASET_ROOT, *DATASET_DIRS]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Notebook root: {PROJECT_ROOT}")
print(f"Dataset directories ensured under: {DATASET_ROOT}")
for path in DATASET_DIRS:
    print(f" - {path.relative_to(PROJECT_ROOT)}")


Notebook root: D:\analiza_snezisc2\1_download_data_and_create_dataset
Dataset directories ensured under: D:\analiza_snezisc2\1_download_data_and_create_dataset\dataset
 - dataset\raw
 - dataset\all_labels
 - dataset\all_labels_probabilities
 - dataset\snow_labels


In [3]:

# -----------------------------------------------------------------------------
# 2. Authenticate and initialize Earth Engine
# -----------------------------------------------------------------------------
EE_PROJECT = None  # replace with your EE project id if needed
try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine client already initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Authenticated and initialized Earth Engine.")

Earth Engine client already initialized.


In [4]:

# -----------------------------------------------------------------------------
# 3. Load region templates and create a sampling plan (~4000 tiles)
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class RegionSeed:
    id: str
    name: str
    bbox: tuple
    collect_start: date
    collect_end: date

    @property
    def width_deg(self) -> float:
        return self.bbox[2] - self.bbox[0]

    @property
    def height_deg(self) -> float:
        return self.bbox[3] - self.bbox[1]


def load_region_seeds(path: Path) -> List[RegionSeed]:
    payload = json.loads(path.read_text("utf-8"))
    regions = []
    for entry in payload["regions"]:
        regions.append(
            RegionSeed(
                id=entry["id"],
                name=entry["name"],
                bbox=tuple(entry["bbox"]),
                collect_start=date.fromisoformat(entry["collect_start"]),
                collect_end=date.fromisoformat(entry["collect_end"]),
            )
        )
    return regions


def km_to_lat_deg(km: float) -> float:
    return km / 110.574


def km_to_lon_deg(km: float, latitude: float) -> float:
    return km / (111.320 * math.cos(math.radians(latitude)))


def sample_bbox(seed: RegionSeed, rng: random.Random, tile_km: float) -> List[float]:
    lat_half = km_to_lat_deg(tile_km / 2.0)
    lat_min = max(seed.bbox[1] + lat_half, seed.bbox[1])
    lat_max = min(seed.bbox[3] - lat_half, seed.bbox[3])
    chosen_lat = rng.uniform(lat_min, lat_max) if lat_min < lat_max else (seed.bbox[1] + seed.bbox[3]) / 2

    lon_half = km_to_lon_deg(tile_km / 2.0, chosen_lat)
    lon_min = max(seed.bbox[0] + lon_half, seed.bbox[0])
    lon_max = min(seed.bbox[2] - lon_half, seed.bbox[2])
    chosen_lon = rng.uniform(lon_min, lon_max) if lon_min < lon_max else (seed.bbox[0] + seed.bbox[2]) / 2

    return [
        chosen_lon - lon_half,
        chosen_lat - lat_half,
        chosen_lon + lon_half,
        chosen_lat + lat_half,
    ]


def sample_date(seed: RegionSeed, rng: random.Random) -> date:
    span_days = (seed.collect_end - seed.collect_start).days
    offset = rng.randint(0, max(span_days, 1))
    return seed.collect_start + timedelta(days=offset)


def make_sampling_plan(seeds: List[RegionSeed], target_tiles: int, rng: random.Random, min_tile_km: float, max_tile_km: float) -> List[Dict[str, object]]:
    plan: List[Dict[str, object]] = []
    if not seeds:
        return plan
    seed_idx = 0
    while len(plan) < target_tiles:
        seed = seeds[seed_idx % len(seeds)]
        seed_idx += 1
        tile_km = rng.uniform(min_tile_km, max_tile_km)
        bbox = sample_bbox(seed, rng, tile_km)
        plan.append(
            {
                "region_id": seed.id,
                "region_name": seed.name,
                "bbox": bbox,
                "acq_date": sample_date(seed, rng),
                "tile_size_km": tile_km,
            }
        )
    return plan


rng = random.Random(RANDOM_SEED)
region_seeds = load_region_seeds(REGION_JSON)
print(f"Loaded {len(region_seeds)} region templates from {REGION_JSON.name}")
sampling_plan = make_sampling_plan(region_seeds, TARGET_TILE_COUNT, rng, MIN_TILE_SIZE_KM, MAX_TILE_SIZE_KM)
print(f"Sampling plan prepared with {len(sampling_plan)} tiles.")
print("Example plan entries:")
for example in sampling_plan[:3]:
    print(example)

Loaded 26000 region templates from bounding_boxes.json
Sampling plan prepared with 20000 tiles.
Example plan entries:
{'region_id': 'coastal_marine_00001', 'region_name': 'Coastal & Marine Influence sample 00001', 'bbox': [-125.12979767051289, 48.183708744722416, -125.09392204268588, 48.20778434114951], 'acq_date': datetime.date(2023, 10, 27), 'tile_size_km': 2.662134999330147}
{'region_id': 'polar_southern_00002', 'region_name': 'Antarctic Fringe sample 00002', 'bbox': [-24.887874745392633, -65.60460722417699, -24.821322056048796, -65.57691869067321], 'acq_date': datetime.date(2023, 8, 1), 'tile_size_km': 3.061631903647672}
{'region_id': 'rural_plains_00003', 'region_name': 'Rural Plains & Agriculture sample 00003', 'bbox': [151.23870956930153, 3.033976642453998, 151.27222853390384, 3.067673920156142], 'acq_date': datetime.date(2023, 4, 15), 'tile_size_km': 3.7260427846369084}


In [5]:

# -----------------------------------------------------------------------------
# 4. Remote sensing helpers (Sentinel-2 + Dynamic World)
# -----------------------------------------------------------------------------
def s2_cloudmask(image: ee.Image) -> Tuple[ee.Image, ee.Image]:
    qa = image.select("QA60")
    cloud_free = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus_free = qa.bitwiseAnd(1 << 11).eq(0)
    return image.updateMask(cloud_free).updateMask(cirrus_free)


def fetch_sentinel_mosaic(
    aoi: ee.Geometry,
    reference_day: date,
    cloud_pct: int,
    initial_window: int = SENTINEL_INITIAL_WINDOW_DAYS,
    max_window: int = SENTINEL_MAX_WINDOW_DAYS,
    step: int = SENTINEL_WINDOW_STEP,
) -> ee.Image:
    window_days = initial_window
    while window_days <= max_window:
        start = (reference_day - timedelta(days=window_days)).isoformat()
        end = (reference_day + timedelta(days=window_days)).isoformat()
        collection = (
            ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
            .map(s2_cloudmask)
        )
        count = collection.size().getInfo()
        if count > 0:
            if window_days > initial_window:
                print(f"Sentinel window expanded to ±{window_days} days (found {count} scenes)")
            return collection.median().select(S2_BANDS).clip(aoi)
        print(f"No Sentinel-2 scenes for ±{window_days} days. Widening window...")
        window_days += step
    raise RuntimeError(
        f"No Sentinel-2 scenes found within ±{max_window} days for AOI {aoi.toGeoJSONString()}"
    )


def fetch_dynamic_world_stack(
    aoi: ee.Geometry,
    reference_day: date,
    initial_window: int = DYNAMIC_WORLD_INITIAL_WINDOW_DAYS,
    max_window: int = DYNAMIC_WORLD_MAX_WINDOW_DAYS,
    step: int = DYNAMIC_WORLD_WINDOW_STEP,
) -> ee.Image:
    window_days = initial_window
    while window_days <= max_window:
        start = (reference_day - timedelta(days=window_days)).isoformat()
        end = (reference_day + timedelta(days=window_days + 1)).isoformat()
        collection = (
            ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
            .filterBounds(aoi)
            .filterDate(start, end)
        )
        count = collection.size().getInfo()
        if count > 0:
            if window_days > initial_window:
                print(f"Dynamic World window expanded to ±{window_days} days (found {count} scenes)")
            label_image = collection.select("label").mode().toUint8().rename("label").clip(aoi)
            prob_image = collection.select(DW_PROBABILITY_BANDS).median().clip(aoi)
            return label_image, prob_image
        print(f"No Dynamic World scenes for ±{window_days} days. Widening window...")
        window_days += step
    raise RuntimeError(
        f"No Dynamic World scenes found within ±{max_window} days for AOI {aoi.toGeoJSONString()}"
    )


def derive_binary_snow_mask(dw_image: ee.Image, min_prob: float) -> ee.Image:
    snow_label = dw_image.select("label").eq(SNOW_CLASS_ID)
    snow_prob = dw_image.select("snow_and_ice").gte(min_prob)
    return snow_label.Or(snow_prob).rename("snow_mask")


def shrink_bbox_to_tile(bbox: list[float], tile_km: float) -> list[float]:
    center_lon = (bbox[0] + bbox[2]) / 2.0
    center_lat = (bbox[1] + bbox[3]) / 2.0
    lat_half = km_to_lat_deg(tile_km / 2.0)
    lon_half = km_to_lon_deg(tile_km / 2.0, center_lat)
    return [
        center_lon - lon_half,
        center_lat - lat_half,
        center_lon + lon_half,
        center_lat + lat_half,
    ]


def compute_tile_scale(tile_km: float, target_px: int = MAX_TILE_PIXELS, base_scale: int = BASE_EXPORT_SCALE) -> int:
    tile_m = tile_km * 1000.0
    required_scale = tile_m / float(target_px)
    safe_scale = max(base_scale, required_scale)
    safe_scale = math.ceil(safe_scale / 5.0) * 5
    return int(max(5, min(safe_scale, 30)))




def has_valid_labels(label_image: ee.Image, region: ee.Geometry, scale: int) -> bool:
    try:
        result = label_image.reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=region,
            scale=scale,
            maxPixels=1_000_000,
        ).getInfo()
    except Exception:
        return False
    if not result:
        return False
    pixels = result.get("label", 0)
    return pixels is not None and pixels > 0

def export_geotiff(image: ee.Image, region: ee.Geometry, target_path: Path, scale: int = 10) -> None:
    target_path.parent.mkdir(parents=True, exist_ok=True)
    geemap.ee_export_image(
        image,
        filename=str(target_path),
        scale=scale,
        region=region,
        file_per_band=False,
        crs="EPSG:4326",
    )


In [ ]:

# -----------------------------------------------------------------------------
# 5. Build the dataset
# -----------------------------------------------------------------------------
def build_dataset(plan: List[Dict[str, object]]) -> None:
    manifest_rows: List[Dict[str, object]] = []
    failures: List[Dict[str, object]] = []

    for idx, tile in enumerate(tqdm(plan, desc="Exporting tiles")):
        tile_id = f"tile_{idx:05d}_{tile['region_id']}"
        bbox = tile["bbox"]
        acquisition: date = tile["acq_date"]

        target_tile_km = min(tile["tile_size_km"], MAX_TILE_SIZE_KM)
        target_bbox = shrink_bbox_to_tile(bbox, target_tile_km)
        aoi = ee.Geometry.Rectangle(target_bbox, proj='EPSG:4326', geodesic=False)

        tile_scale = compute_tile_scale(target_tile_km)
        if tile_scale > BASE_EXPORT_SCALE:
            print(f"Adaptive scale {tile_scale} m used for {tile_id} to cap tiles at {MAX_TILE_PIXELS} px.")

        sentinel_path = RAW_DIR / f"{tile_id}_sentinel.tif"
        dw_label_path = ALL_LABEL_DIR / f"{tile_id}_dynamic_world_label.tif"
        dw_prob_path = ALL_LABEL_PROB_DIR / f"{tile_id}_dynamic_world_prob.tif"
        snow_path = SNOW_LABEL_DIR / f"{tile_id}_snow_mask.tif"

        try:
            dw_label_img, dw_prob_img = fetch_dynamic_world_stack(aoi, acquisition)
            if not has_valid_labels(dw_label_img, aoi, tile_scale):
                msg = "Dynamic World label image contains no valid pixels"
                failures.append({"tile_id": tile_id, "region_id": tile["region_id"], "error": msg})
                print(f"⚠️ Skipping {tile_id}: {msg}")
                continue
        
            sentinel_img = fetch_sentinel_mosaic(aoi, acquisition, SENTINEL_CLOUD_THRESHOLD)
            combined_dw = dw_label_img.addBands(dw_prob_img)
            snow_img = derive_binary_snow_mask(combined_dw, SNOW_MIN_PROB)
            
            export_geotiff(sentinel_img, aoi, sentinel_path, scale=tile_scale)
            export_geotiff(dw_label_img, aoi, dw_label_path, scale=tile_scale)
            export_geotiff(dw_prob_img, aoi, dw_prob_path, scale=tile_scale)
            export_geotiff(snow_img, aoi, snow_path, scale=tile_scale)
            
            manifest_rows.append(
                    {
                        "tile_id": tile_id,
                        "region_id": tile["region_id"],
                        "region_name": tile["region_name"],
                        "acquisition_date": acquisition.isoformat(),
                        "bbox_min_lon": target_bbox[0],
                        "bbox_min_lat": target_bbox[1],
                        "bbox_max_lon": target_bbox[2],
                        "bbox_max_lat": target_bbox[3],
                        "tile_size_km": target_tile_km,
                        "sentinel_path": str(sentinel_path.relative_to(DATASET_ROOT)),
                        "dynamic_world_label_path": str(dw_label_path.relative_to(DATASET_ROOT)),
                        "dynamic_world_prob_path": str(dw_prob_path.relative_to(DATASET_ROOT)),
                        "snow_mask_path": str(snow_path.relative_to(DATASET_ROOT)),
                    }
                )
        except Exception as exc:  # noqa: BLE001
            failures.append(
                {
                    "tile_id": tile_id,
                    "region_id": tile["region_id"],
                    "error": str(exc),
                }
            )
            print(f"⚠️ Skipping {tile_id}: {exc}")

    if manifest_rows:
        with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as handle:
            fieldnames = list(manifest_rows[0].keys())
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(manifest_rows)
        print(f"Manifest saved to {MANIFEST_PATH}")

    if failures:
        FAILED_LOG.write_text(json.dumps(failures, indent=2), encoding='utf-8')
        print(f"Failure log saved to {FAILED_LOG} ({len(failures)} skipped tiles)")
    else:
        print("All tiles exported successfully.")


# Uncomment to start the exports once the environment is ready.
build_dataset(sampling_plan)

Exporting tiles:   0%|          | 0/20000 [00:00<?, ?it/s]

Generating URL ...
Please wait ...
Data downloaded to D:\analiza_snezisc2\1_download_data_and_create_dataset\dataset\raw\tile_00000_coastal_marine_00001_sentinel.tif
Generating URL ...
Please wait ...
Data downloaded to D:\analiza_snezisc2\1_download_data_and_create_dataset\dataset\all_labels\tile_00000_coastal_marine_00001_dynamic_world_label.tif
Generating URL ...
Please wait ...
Data downloaded to D:\analiza_snezisc2\1_download_data_and_create_dataset\dataset\all_labels_probabilities\tile_00000_coastal_marine_00001_dynamic_world_prob.tif
Generating URL ...
Please wait ...
Data downloaded to D:\analiza_snezisc2\1_download_data_and_create_dataset\dataset\snow_labels\tile_00000_coastal_marine_00001_snow_mask.tif
No Dynamic World scenes for ±10 days. Widening window...
No Dynamic World scenes for ±20 days. Widening window...
No Dynamic World scenes for ±30 days. Widening window...
⚠️ Skipping tile_00001_polar_southern_00002: Cannot convert a computed geometry to GeoJSON. Wrap a getInfo(

## 6. Next steps

1. Double-check that `sampling_plan` looks reasonable (enough geographic diversity and bounding boxes within the target regions).
2. Uncomment and run `build_dataset(sampling_plan)` to start exporting ~4k tiles. Expect the job to take several hours; lower `TARGET_TILE_COUNT` for trial runs.
3. Inspect samples under `dataset/raw`, `dataset/all_labels`, and `dataset/snow_labels` (e.g., using QGIS) to ensure the Sentinel imagery and labels align.
4. Feed `dataset/manifest.csv` into your ML pipeline to pair Sentinel features with snow masks.